# Module 9 · 音訊 2：波形 → 張量前處理（2026 音訊管線的起點）

> **本節定位｜2026 主流核心**
> 預訓練音訊模型（Whisper、wav2vec2、HuBERT）對輸入有嚴格要求：
> **固定取樣率（多為 16kHz）、單聲道、正規化**，且多數吃 **log-mel 頻譜**。
> 本節把原始音訊標準化成模型要的張量。

## 學習目標
- 掌握音訊前處理三步：**重採樣到 16kHz → 轉單聲道 → 振幅正規化**。
- 把波形轉成 **log-mel 頻譜**，理解其形狀 `(n_mels, T)`。
- 釐清「資料結構」：波形 `(N, samples)` 與頻譜 `(N, n_mels, T)`。

<!-- concept-image:02_audio_to_tensor_fig1 -->
<div align="center">
<img src="concept_images/02_audio_to_tensor_fig1_preprocessing_20260611_221708.png" alt="音訊前處理三步驟" width="640" style="max-width:100%;">
<br><sub>圖 1 · 音訊前處理三步驟</sub>
</div>

## 0. 資料結構設計：音訊張量與標籤

| 物件 | 形狀 | 說明 |
|:--|:--|:--|
| 原始波形 | `(samples,)` 或 `(channels, samples)` | 取樣率決定時間解析度 |
| 標準化波形 | `(N, samples)` | 16kHz、單聲道、正規化後的批次 |
| log-mel 頻譜 | `(N, n_mels, T)` | 多數神經模型的實際輸入（n_mels≈80） |
| 標籤（分類） | `(N,)` | 類別索引（如聲音事件） |
| 標籤（ASR） | `list[str]` | 逐句轉錄文字（再經 tokenizer） |

<!-- concept-image:02_audio_to_tensor_fig2 -->
<div align="center">
<img src="concept_images/02_audio_to_tensor_fig2_structures_20260611_221743.png" alt="波形與 log-mel 頻譜的資料結構" width="640" style="max-width:100%;">
<br><sub>圖 2 · 波形與 log-mel 頻譜的資料結構</sub>
</div>

In [ ]:
import numpy as np
import librosa
# 真實音訊，以「原生取樣率」載入（不重採樣），模擬拿到一段任意取樣率的錄音
wave_native, sr_orig = librosa.load(librosa.example("trumpet"), sr=None, duration=3.0)
wave_native = wave_native.astype(np.float32)
print(f"真實音訊 librosa 'trumpet' 原始波形: {wave_native.shape}  原生取樣率 {sr_orig}Hz")

## 1. 重採樣到 16kHz + 正規化（torchaudio）

預訓練模型多以 16kHz 訓練；不同來源的音訊取樣率不一，**必須統一重採樣**。
需 `multimodal` extra：`uv sync --extra multimodal`。

In [ ]:
try:
    import torch, torchaudio
    wav = torch.from_numpy(wave_native).unsqueeze(0)       # (1, samples) = (channel, samples)
    resampler = torchaudio.transforms.Resample(orig_freq=sr_orig, new_freq=16000)
    wav16 = resampler(wav)                                 # (1, samples')
    # 若是多聲道，轉單聲道：對 channel 維取平均
    wav_mono = wav16.mean(dim=0, keepdim=True)
    # 振幅正規化到 [-1, 1]
    wav_norm = wav_mono / wav_mono.abs().max().clamp(min=1e-9)
    print(f"重採樣後: {tuple(wav_norm.shape)}  取樣率 16000Hz")
    print(f"長度: {wav_norm.shape[-1]} samples = {wav_norm.shape[-1]/16000:.2f} 秒")
    print(f"振幅範圍: [{wav_norm.min():.2f}, {wav_norm.max():.2f}]")
    HAS_TA = True
except Exception as e:
    HAS_TA = False
    print("未安裝 torchaudio，請 `uv sync --extra multimodal`。說明仍可閱讀。", e)

## 2. 波形 → log-mel 頻譜

log-mel 頻譜是 2026 多數神經音訊模型的實際輸入：先做 mel 頻譜，再取 log（壓縮動態範圍）。

<!-- concept-image:02_audio_to_tensor_fig3 -->
<div align="center">
<img src="concept_images/02_audio_to_tensor_fig3_batching_20260611_221817.png" alt="批次化張量形狀與 Padding" width="640" style="max-width:100%;">
<br><sub>圖 3 · 批次化張量形狀與 Padding</sub>
</div>

In [ ]:
if HAS_TA:
    mel = torchaudio.transforms.MelSpectrogram(sample_rate=16000, n_fft=400,
                                               hop_length=160, n_mels=80)
    m = mel(wav_norm)                                      # (1, n_mels, T)
    log_mel = torch.log(m + 1e-6)
    print(f"log-mel 頻譜: {tuple(log_mel.shape)}  (channel, n_mels, T)")
    print(f"→ 批次化後即 (N, n_mels, T)，n_mels={log_mel.shape[1]}，時間幀 T={log_mel.shape[2]}")

## 3. 批次化的形狀

不同長度的音訊，批次時需 **pad 到等長**（或裁成固定窗），再堆疊。

In [ ]:
if HAS_TA:
    a = wav_norm.squeeze(0)
    b = wav_norm.squeeze(0)[:8000]                         # 較短的一段
    L = max(a.shape[-1], b.shape[-1])
    def pad_to(x, L):
        return torch.nn.functional.pad(x, (0, L - x.shape[-1]))
    batch = torch.stack([pad_to(a, L), pad_to(b, L)])      # (N, samples)
    print(f"波形批次（pad 後）: {tuple(batch.shape)}  (N, samples)")

## 小結

| 重點 | 內容 |
|:--|:--|
| 三步前處理 | 重採樣 16kHz → 單聲道 → 振幅正規化 |
| 神經模型輸入 | log-mel 頻譜 `(N, n_mels, T)`，n_mels≈80 |
| 波形批次 | `(N, samples)`，需 pad 等長 |
| 標籤 | 分類 `(N,)`；ASR 為文字（再 tokenizer） |

**下一節 `03_modern_audio_representations`**：用 HuggingFace `AutoFeatureExtractor`
直接產出 Whisper / wav2vec2 期望的輸入，並取得音訊嵌入。